In [ ]:
import pandas as pd
import numpy as np
import iqplot
from itertools import combinations


import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')


In [ ]:
df = pd.read_csv('single_subject_post_freezethaw.csv')
df['parent_subjects'].unique()

In [ ]:
df

In [ ]:
df_in = df.loc[df['is_inoculumn'],:]
df_in = df_in.loc[df_in['relative_abundance'] >=1e-2,:]

df_in['mesocosm'].unique()

In [ ]:
c1s = []
c2s = []
same_media = []
same_sub = []
shared_sps = []
for pair in combinations(df_in['mesocosm'].unique(),2):
    c1,c2 = pair
    dfc1 = df_in.loc[df['mesocosm'] == c1,:]
    
    dfc2 = df_in.loc[df['mesocosm'] == c2,:]
    sp_shared = len(np.intersect1d(dfc1['species_id'].unique(), dfc2['species_id'].unique()))
    shared_sps.append(sp_shared)
   # print(c1,c2,len(dfc1['species_id'].unique()), len(dfc2['species_id'].unique()))
   # print(sp_shared)
    c1_sub = c1.split('-')[1]
    c1_media = c1.split('-')[-2]
    c1s.append(c1)
    c2s.append(c2)
    
    c2_sub = c2.split('-')[1]
    c2_media = c2.split('-')[-2]
 #   shared_media = c1_media == c2_media
    same_media.append(c1_media == c2_media)
    same_sub.append(c1_sub == c2_sub)
  #  shared_sub = c1_sub == c2_sub
    
   # print(c1_sub, c2_sub)
dfgood = pd.DataFrame(data={'c1':c1s,'c2':c2s,'same_media':same_media, 'same_sub':same_sub, 'shared_sps': shared_sps})
dfgood['same_media'] = dfgood['same_media'] #.astype(str)
dfgood['same_sub'] = dfgood['same_sub'] #.astype(str)
dfgood

In [ ]:
dfgood['counts']=1
dfgood_gr = dfgood.groupby(['same_media','same_sub', 'shared_sps']).sum().reset_index()
cmap = bokeh.palettes.Category10[10]
i = 0
p = bokeh.plotting.figure(height = 200, width = 400)
for med in ['True',]:
    dfgood_gr1 = dfgood_gr.loc[dfgood_gr['same_media']==med,:]
    for sub in ['False']:
        dfgood_gr2 = dfgood_gr1.loc[dfgood_gr1['same_sub']==sub,:]
        p.circle(dfgood_gr2['shared_sps'].values, dfgood_gr2['counts'].values, color = cmap[i], ) #legend_label = f'{str(med)} and {str(sub)}')
        p.ray(x = dfgood_gr2['shared_sps'].values, y = dfgood_gr2['counts'].values, angle = -np.pi/2, color = cmap[i], line_width = 2)
        i = i + 1 

In [ ]:
p.y_range = bokeh.models.Range1d(-.05,3.05)
p.x_range = bokeh.models.Range1d(-.05,11.05)
p.title.text= 'Num of shared species above 1 %' 
p.xaxis.axis_label= 'Number of shared species'
p.yaxis.axis_label= 'Count'
bokeh.io.show(p)

In [ ]:
dfgood['same_media'] = dfgood['same_media'].astype(str)
dfgood['same_sub'] = dfgood['same_sub'].astype(str)
p_spike = iqplot.spike(data=dfgood, q="shared_sps",cats = ['same_sub','same_media',],fraction=True)

bokeh.io.show(p_spike)

In [ ]:
dfgoodb = dfgood.loc[dfgood['same_media']==True,:]
dfgoodb = dfgoodb.loc[dfgoodb['same_sub']==True,:]
